# Video Swin -- Out-of-Distribution Evaluation

Evaluates the trained Basketball-51 baseline (`runs/swin3d_b51_<variant>/best.pt`) on **out-of-distribution** footage: a high-school girls basketball game. The model was trained exclusively on NBA broadcast clips, so this set is OOD along several axes simultaneously (player demographic, gym, broadcast quality, camera operator, court markings).

**Pipeline.** Same preprocessing as the in-domain game-clips evaluation:

1. **FPS normalization** to 24 FPS for any source faster than that.
2. **Spatial downscaling** to QVGA (320 x 240) using a centred 4:3 crop, to match Basketball-51's native resolution.
3. Video Swin standard temporal sampling (32 frames uniformly) and spatial resize to 224 x 224.

**Metrics.**

- Top-1 accuracy (8-class)
- Shot-type accuracy (4-way: ft / 2p / mp / 3p)
- Make/miss accuracy (binary)
- Per-class precision / recall / F1
- Three confusion matrices (8-class, 4-class shot-type, 2-class make/miss)
- Score counter (total predicted vs ground-truth points)

All artefacts are written under `PROJECT_ROOT/eval_ood/`.

## 1. Setup

In [2]:
from pathlib import Path

try:
    from google.colab import drive
    if not Path('/content/drive').exists() or not any(Path('/content/drive').iterdir()):
        drive.mount('/content/drive', force_remount=False)
    IN_COLAB = True
except Exception:
    IN_COLAB = False

PROJECT_ROOT = Path('/content/drive/MyDrive/Courses/Spring 1/Applied Computer Vision/Project/Test') if IN_COLAB else Path('.').resolve()
RUNS_ROOT    = PROJECT_ROOT / 'runs'
VARIANT      = 'base'

CKPT_PATH      = RUNS_ROOT / VARIANT / 'best.pt'
OOD_ZIP_PATH   = Path('/content/drive/MyDrive/Courses/Spring 1/Applied Computer Vision/Project/Datasets/OOD_data.zip')    # <-- set this to your zip
OOD_RAW_ROOT   = PROJECT_ROOT / 'ood_raw'
OOD_PRE_ROOT   = PROJECT_ROOT / 'ood_preprocessed'
REPORT_DIR     = PROJECT_ROOT / 'eval_ood'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT  :', PROJECT_ROOT)
print('CKPT_PATH     :', CKPT_PATH,    '  exists:', CKPT_PATH.exists())
print('OOD_ZIP_PATH  :', OOD_ZIP_PATH, '  exists:', OOD_ZIP_PATH.exists())
print('OOD_RAW_ROOT  :', OOD_RAW_ROOT)
print('OOD_PRE_ROOT  :', OOD_PRE_ROOT)
print('REPORT_DIR    :', REPORT_DIR)

assert CKPT_PATH.exists(),    f'Trained checkpoint not found at {CKPT_PATH}.'
assert OOD_ZIP_PATH.exists(), f'OOD zip not found at {OOD_ZIP_PATH}. Set OOD_ZIP_PATH above.'

PROJECT_ROOT  : /content/drive/MyDrive/Courses/Spring 1/Applied Computer Vision/Project/Test
CKPT_PATH     : /content/drive/MyDrive/Courses/Spring 1/Applied Computer Vision/Project/Test/runs/base/best.pt   exists: True
OOD_ZIP_PATH  : /content/drive/MyDrive/Courses/Spring 1/Applied Computer Vision/Project/Datasets/OOD_data.zip   exists: True
OOD_RAW_ROOT  : /content/drive/MyDrive/Courses/Spring 1/Applied Computer Vision/Project/Test/ood_raw
OOD_PRE_ROOT  : /content/drive/MyDrive/Courses/Spring 1/Applied Computer Vision/Project/Test/ood_preprocessed
REPORT_DIR    : /content/drive/MyDrive/Courses/Spring 1/Applied Computer Vision/Project/Test/eval_ood


In [3]:
import sys, subprocess
if IN_COLAB:
    pkgs = ['decord', 'scikit-learn']
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=False)
    if (PROJECT_ROOT / 'setup.py').exists():
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_ROOT)], check=False)
print('Dependencies ready.')

Dependencies ready.


## 2. Class taxonomy

In [7]:
import numpy as np
sys.path.insert(0, str(PROJECT_ROOT)) # Add PROJECT_ROOT to sys.path to find local modules
from videoswin import BASKETBALL51_CLASSES
from videoswin.data.taxonomy import (
    SHOT_TYPES, OUTCOMES,
    SHOT_TYPE_ID_PER_CLASS, OUTCOME_ID_PER_CLASS, POINTS_PER_CLASS,
)

CLASS_NAMES = list(BASKETBALL51_CLASSES)
N_CLASSES   = len(CLASS_NAMES)

# 8-class display order: free throw -> 2pt -> mid-range -> 3pt, miss/make pairs
ORDER     = ('ft0', 'ft1', '2p0', '2p1', 'mp0', 'mp1', '3p0', '3p1')
ORDER_IDX = [CLASS_NAMES.index(c) for c in ORDER]

points_vec = POINTS_PER_CLASS.astype(np.float32)

def softmax(x, axis=1):
    x = x - x.max(axis=axis, keepdims=True)
    ex = np.exp(x); return ex / ex.sum(axis=axis, keepdims=True)

print('8-class labels :', CLASS_NAMES)
print('Shot types     :', list(SHOT_TYPES))
print('Outcomes       :', list(OUTCOMES))

8-class labels : ['2p0', '2p1', '3p0', '3p1', 'ft0', 'ft1', 'mp0', 'mp1']
Shot types     : ['2p', '3p', 'ft', 'mp']
Outcomes       : ['miss', 'make']


In [8]:
# Data loading

import zipfile, shutil
from pathlib import Path
import pandas as pd
import cv2

if OOD_RAW_ROOT.exists() and any(OOD_RAW_ROOT.iterdir()):
    print(f'{OOD_RAW_ROOT} already populated -- skipping extraction.')
else:
    OOD_RAW_ROOT.mkdir(parents=True, exist_ok=True)
    print(f'Extracting {OOD_ZIP_PATH.name} -> {OOD_RAW_ROOT} ...')
    with zipfile.ZipFile(OOD_ZIP_PATH) as zf:
        zf.extractall(OOD_RAW_ROOT)
    print('Extraction done.')

def _locate_class_root(root: Path) -> Path:
    if any((root / c).is_dir() for c in CLASS_NAMES):
        return root
    candidates = [p for p in root.iterdir() if p.is_dir()]
    for cand in candidates:
        if any((cand / c).is_dir() for c in CLASS_NAMES):
            return cand
    raise RuntimeError(f'No 8-class folder layout found under {root}. '
                       f'Expected at least one of {CLASS_NAMES} as a subfolder.')

OOD_CLASS_ROOT = _locate_class_root(OOD_RAW_ROOT)
print(f'OOD class root: {OOD_CLASS_ROOT}')

VALID_EXT = {'.mp4', '.mov', '.MP4', '.MOV', '.avi', '.AVI'}
rows = []
for cls in CLASS_NAMES:
    cls_dir = OOD_CLASS_ROOT / cls
    if not cls_dir.exists():
        continue
    for p in sorted(cls_dir.iterdir()):
        if p.suffix not in VALID_EXT:
            continue
        cap = cv2.VideoCapture(str(p))
        ok = cap.isOpened()
        fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
        n   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
        h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
        cap.release()
        rows.append({
            'class': cls, 'name': p.name,
            'fps': round(fps, 2), 'frames': n,
            'duration_s': round(n / fps, 2) if fps > 0 else None,
            'width': w, 'height': h,
            'rel_path': f'{cls}/{p.name}', 'src_path': str(p),
            'status': 'ok' if ok and n > 0 else 'cv2-open-failed',
        })
df_src = pd.DataFrame(rows)
if df_src.empty:
    raise RuntimeError(f'No video files discovered under {OOD_CLASS_ROOT}.')

print(f'Discovered {len(df_src)} OOD clips.')
print('\nClass distribution:')
print(df_src['class'].value_counts().reindex(CLASS_NAMES, fill_value=0).to_string())
uniq_res = sorted({(int(w), int(h)) for w, h in df_src[['width', 'height']].itertuples(index=False, name=None) if w and h})
uniq_fps = sorted({float(f) for f in df_src['fps'].dropna().tolist() if f})
print(f'\nSource resolutions: {uniq_res}')
print(f'Source FPS values : {uniq_fps}')

/content/drive/MyDrive/Courses/Spring 1/Applied Computer Vision/Project/Test/ood_raw already populated -- skipping extraction.
OOD class root: /content/drive/MyDrive/Courses/Spring 1/Applied Computer Vision/Project/Test/ood_raw/Real World
Discovered 11 OOD clips.

Class distribution:
class
2p0    2
2p1    3
3p0    1
3p1    1
ft0    0
ft1    2
mp0    2
mp1    0

Source resolutions: [(640, 360)]
Source FPS values : [30.0]


## 4. Preprocessing -- FPS normalisation + QVGA conversion


In [10]:
import cv2
import numpy as np
from pathlib import Path

TARGET_FPS = 24
TARGET_W   = 320
TARGET_H   = 240

def preprocess_clip(src_path, dst_path,
                    target_fps: int = TARGET_FPS,
                    target_w: int = TARGET_W,
                    target_h: int = TARGET_H) -> dict:
    src_path = Path(src_path); dst_path = Path(dst_path)
    cap = cv2.VideoCapture(str(src_path))
    if not cap.isOpened():
        raise RuntimeError(f'cv2 could not open {src_path}')
    src_fps = cap.get(cv2.CAP_PROP_FPS) or float(target_fps)
    frames = []
    while True:
        ok, fr = cap.read()
        if not ok:
            break
        frames.append(fr)
    cap.release()
    if not frames:
        raise RuntimeError(f'No frames decoded from {src_path}')
    n_in = len(frames); H, W = frames[0].shape[:2]
    src_fps_eff = src_fps if src_fps > 0 else float(target_fps)
    if src_fps_eff > target_fps:
        duration_s = n_in / src_fps_eff
        n_out = max(1, int(round(duration_s * target_fps)))
        idx = np.clip(np.linspace(0, n_in - 1, n_out).round().astype(int), 0, n_in - 1)
        frames = [frames[i] for i in idx]
        out_fps = float(target_fps)
    else:
        out_fps = float(src_fps_eff)
    target_aspect = target_w / target_h
    if W / H > target_aspect:
        new_w = int(round(H * target_aspect)); x0 = (W - new_w) // 2
        y0    = 0; new_h = H
    elif W / H < target_aspect:
        new_h = int(round(W / target_aspect)); y0 = (H - new_h) // 2
        x0    = 0; new_w = W
    else:
        new_w, new_h, x0, y0 = W, H, 0, 0
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(str(dst_path), fourcc, out_fps, (target_w, target_h))
    if not writer.isOpened():
        raise RuntimeError(f'Could not open VideoWriter for {dst_path}')
    try:
        for fr in frames:
            crop = fr[y0:y0 + new_h, x0:x0 + new_w]
            small = cv2.resize(crop, (target_w, target_h), interpolation=cv2.INTER_AREA)
            writer.write(small)
    finally:
        writer.release()
    return {'src_fps': src_fps, 'out_fps': out_fps,
            'src_n_frames': n_in, 'kept_n_frames': len(frames),
            'src_size': (W, H), 'crop_box': (int(x0), int(y0), int(new_w), int(new_h))}

## 5. Batch preprocess

In [11]:
import time
from pathlib import Path

OOD_PRE_ROOT.mkdir(parents=True, exist_ok=True)
log_rows = []
skipped = converted = failed = 0

for _, row in df_src.iterrows():
    if row['status'] != 'ok':
        log_rows.append({**row.to_dict(), 'pp_status': 'skip-bad-source'})
        skipped += 1
        continue
    src = Path(row['src_path'])
    dst = OOD_PRE_ROOT / row['class'] / (src.stem + '.mp4')
    if dst.exists() and dst.stat().st_size > 0:
        log_rows.append({**row.to_dict(), 'pp_status': 'skip-exists', 'dst_path': str(dst)})
        skipped += 1
        continue
    t0 = time.time()
    try:
        meta = preprocess_clip(src, dst)
        log_rows.append({
            **row.to_dict(), 'pp_status': 'ok',
            'dst_path': str(dst),
            'pp_kept_frames': meta['kept_n_frames'],
            'pp_out_fps':     meta['out_fps'],
            'pp_elapsed_s':   round(time.time() - t0, 3),
        })
        converted += 1
    except Exception as e:
        log_rows.append({**row.to_dict(), 'pp_status': f'fail: {e}', 'dst_path': str(dst)})
        failed += 1

df_pp = pd.DataFrame(log_rows)
df_pp.to_csv(REPORT_DIR / 'preprocessing_log.csv', index=False)
print(f'Converted: {converted}   Skipped: {skipped}   Failed: {failed}')
print(f'Wrote {REPORT_DIR/"preprocessing_log.csv"}')

Converted: 11   Skipped: 0   Failed: 0
Wrote /content/drive/MyDrive/Courses/Spring 1/Applied Computer Vision/Project/Test/eval_ood/preprocessing_log.csv


## 6. Run inference

Builds a `Basketball51Dataset` over the preprocessed OOD clips and runs the trained baseline checkpoint over it once. Cached predictions are stored in `eval_ood/raw_predictions.npz`.

In [12]:
import torch, json
from torch.utils.data import DataLoader
import numpy as np
from videoswin.data import Basketball51Dataset, VideoEval, worker_init_fn
from videoswin.eval import collect_predictions
from videoswin.models import build_video_swin
from videoswin.utils import load_checkpoint

PREDICTIONS_NPZ = REPORT_DIR / 'raw_predictions.npz'
FORCE_REBUILD   = False

samples = []
for cls_idx, cls in enumerate(CLASS_NAMES):
    cls_dir = OOD_PRE_ROOT / cls
    if not cls_dir.exists():
        continue
    for p in sorted(cls_dir.iterdir()):
        if p.suffix.lower() in {'.mp4', '.mov'} and p.stat().st_size > 0:
            samples.append((f'{cls}/{p.name}', cls_idx))
assert samples, f'No preprocessed clips under {OOD_PRE_ROOT}. Run section 5 first.'
n_classes_present = len({CLASS_NAMES[c] for _, c in samples})
print(f'Inference set: {len(samples)} clips across {n_classes_present} classes')

if PREDICTIONS_NPZ.exists() and not FORCE_REBUILD:
    print(f'Loading cached predictions from {PREDICTIONS_NPZ}')
    npz = np.load(PREDICTIONS_NPZ, allow_pickle=False)
    logits       = npz['logits']
    labels       = npz['labels']
    sample_paths = [str(p) for p in npz['sample_paths']]
else:
    sample_paths = [str(OOD_PRE_ROOT / rel_path) for rel_path, _ in samples]
    dataset = Basketball51Dataset(
        root=OOD_PRE_ROOT, samples=samples,
        transform=VideoEval(out_h=224, out_w=224),
        num_frames=32, temporal_jitter=False,
    )
    loader = DataLoader(dataset, batch_size=4, shuffle=False, num_workers=4,
                        pin_memory=True, worker_init_fn=worker_init_fn)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Loading checkpoint from {CKPT_PATH} on device={device} ...')
    model = build_video_swin(num_classes=N_CLASSES, variant=VARIANT).to(device)
    load_checkpoint(CKPT_PATH, model, map_location=device)
    model.eval()
    out = collect_predictions(model, loader, device, amp=True, collect_features=False)
    logits = out['logits']
    labels = out['labels']
    np.savez_compressed(
        PREDICTIONS_NPZ,
        logits=logits.astype(np.float32),
        labels=labels.astype(np.int64),
        sample_paths=np.array(sample_paths),
    )
    print(f'Cached predictions at {PREDICTIONS_NPZ}')

probs = softmax(logits, axis=1)
preds = probs.argmax(axis=1)
N     = probs.shape[0]

true_shot_type = SHOT_TYPE_ID_PER_CLASS[labels]
pred_shot_type = SHOT_TYPE_ID_PER_CLASS[preds]
true_outcome   = OUTCOME_ID_PER_CLASS[labels]
pred_outcome   = OUTCOME_ID_PER_CLASS[preds]
true_points    = points_vec[labels]
pred_points    = points_vec[preds]

overall_acc   = (preds == labels).mean()
shot_type_acc = (pred_shot_type == true_shot_type).mean()
outcome_acc   = (pred_outcome   == true_outcome).mean()
print(f'\nClips                 : {N}')
print(f'Top-1 (8-class) acc   : {overall_acc*100:6.2f}%')
print(f'Shot-type acc (4-way) : {shot_type_acc*100:6.2f}%')
print(f'Make/miss acc (binary): {outcome_acc*100:6.2f}%')

Inference set: 11 clips across 6 classes
Loading checkpoint from /content/drive/MyDrive/Courses/Spring 1/Applied Computer Vision/Project/Test/runs/base/best.pt on device=cuda ...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Downloading: "https://download.pytorch.org/models/swin3d_b_22k-7c6ae6fa.pth" to /root/.cache/torch/hub/checkpoints/swin3d_b_22k-7c6ae6fa.pth


100%|██████████| 364M/364M [00:07<00:00, 47.8MB/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/content/drive/MyDrive/Courses/Spring 1/Applied Computer Vision/Project/Test/videoswin/eval.py:95: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Cached predictions at /content/drive/MyDrive/Courses/Spring 1/Applied Computer Vision/Project/Test/eval_ood/raw_predictions.npz

Clips                 : 11
Top-1 (8-class) acc   :  45.45%
Shot-type acc (4-way) :  54.55%
Make/miss acc (binary):  72.73%


## 7. Headline metrics

In [13]:
headline = pd.DataFrame([
    ['Number of clips',                f'{N}'],
    ['Top-1 accuracy (8-class)',       f'{overall_acc*100:6.2f}%'],
    ['Shot-type accuracy (4-way)',     f'{shot_type_acc*100:6.2f}%'],
    ['Make/miss accuracy (binary)',    f'{outcome_acc*100:6.2f}%'],
    ['Random-guess top-1 (uniform)',   f'{100.0/N_CLASSES:6.2f}%'],
    ['Majority-class top-1',           f'{(np.bincount(labels, minlength=N_CLASSES) / max(N,1)).max()*100:6.2f}%'],
], columns=['Metric', 'Value'])
headline.to_csv(REPORT_DIR / 'headline_metrics.csv', index=False)
try:
    from IPython.display import display
    display(headline)
except Exception:
    print(headline.to_string(index=False))
print(f'\nWrote {REPORT_DIR/"headline_metrics.csv"}')

,Metric,Value
0,Number of clips,11
1,Top-1 accuracy (8-class),45.45%
2,Shot-type accuracy (4-way),54.55%
3,Make/miss accuracy (binary),72.73%
4,Random-guess top-1 (uniform),12.50%
5,Majority-class top-1,27.27%



Wrote /content/drive/MyDrive/Courses/Spring 1/Applied Computer Vision/Project/Test/eval_ood/headline_metrics.csv


## 8. Per-class precision / recall / F1

In [14]:
from sklearn.metrics import precision_recall_fscore_support

p, r, f1, sup = precision_recall_fscore_support(
    labels, preds, labels=list(range(N_CLASSES)), zero_division=0,
)
per_class_acc = np.array([
    (preds[labels == c] == c).mean() if (labels == c).sum() else float('nan')
    for c in range(N_CLASSES)
])
df_pc = pd.DataFrame({
    'class':      CLASS_NAMES,
    'support':    sup,
    'accuracy%':  (per_class_acc * 100).round(2),
    'precision%': (np.array(p)  * 100).round(2),
    'recall%':    (np.array(r)  * 100).round(2),
    'f1%':        (np.array(f1) * 100).round(2),
}).set_index('class').loc[list(ORDER)].reset_index()
df_pc.loc['MACRO'] = {
    'class': 'MACRO',
    'support':    int(df_pc['support'].sum()),
    'accuracy%':  round(np.nanmean(df_pc['accuracy%']),  2),
    'precision%': round(df_pc['precision%'].mean(), 2),
    'recall%':    round(df_pc['recall%'].mean(),    2),
    'f1%':        round(df_pc['f1%'].mean(),        2),
}
df_pc.to_csv(REPORT_DIR / 'per_class_metrics.csv', index=False)
try:
    display(df_pc)
except Exception:
    print(df_pc.to_string(index=False))
print(f'\nWrote {REPORT_DIR/"per_class_metrics.csv"}')

,class,support,accuracy%,precision%,recall%,f1%
0,ft0,0,NaN,0.00,0.00,0.00
1,ft1,2,50.00,100.00,50.00,66.67
2,2p0,2,0.00,0.00,0.00,0.00
3,2p1,3,33.33,100.00,33.33,50.00
4,mp0,2,50.00,50.00,50.00,50.00
5,mp1,0,NaN,0.00,0.00,0.00
6,3p0,1,100.00,33.33,100.00,50.00
7,3p1,1,100.00,100.00,100.00,100.00
MACRO,MACRO,11,55.56,47.92,41.67,39.58



Wrote /content/drive/MyDrive/Courses/Spring 1/Applied Computer Vision/Project/Test/eval_ood/per_class_metrics.csv


## 9. Confusion matrices

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

def _plot_cm(cm_counts, class_names, title, out_path, figsize=(7, 6), annot_fontsize=10):
    cm_pct = cm_counts.astype(np.float64) / np.clip(cm_counts.sum(axis=1, keepdims=True), 1, None) * 100.0
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(cm_pct, cmap='Blues', vmin=0, vmax=100)
    ax.set_xticks(range(len(class_names))); ax.set_xticklabels(class_names, rotation=45, ha='right')
    ax.set_yticks(range(len(class_names))); ax.set_yticklabels(class_names)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(title)
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            if cm_counts[i, j] == 0 and cm_pct[i, j] == 0:
                continue
            ax.text(j, i, f'{cm_pct[i, j]:.0f}\n(n={cm_counts[i, j]})',
                    ha='center', va='center',
                    color=('white' if cm_pct[i, j] > 50 else 'black'),
                    fontsize=annot_fontsize)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='% of true class')
    fig.tight_layout(); fig.savefig(out_path, dpi=160, bbox_inches='tight')
    plt.show(); plt.close(fig)

labels_ord = np.array([ORDER_IDX.index(int(l)) for l in labels])
preds_ord  = np.array([ORDER_IDX.index(int(p)) for p in preds])
cm_8 = confusion_matrix(labels_ord, preds_ord, labels=list(range(len(ORDER))))
_plot_cm(cm_8, list(ORDER), 'OOD: 8-class confusion (row-normalised %)',
         REPORT_DIR / 'confusion_matrix_8x8.png',
         figsize=(8.5, 7.5), annot_fontsize=9)

cm_shot = confusion_matrix(true_shot_type, pred_shot_type, labels=list(range(len(SHOT_TYPES))))
_plot_cm(cm_shot, list(SHOT_TYPES), 'OOD: shot-type confusion (4-way, %)',
         REPORT_DIR / 'confusion_matrix_shot_type.png', figsize=(6, 5))

cm_mm = confusion_matrix(true_outcome, pred_outcome, labels=list(range(len(OUTCOMES))))
_plot_cm(cm_mm, list(OUTCOMES), 'OOD: make/miss confusion (binary, %)',
         REPORT_DIR / 'confusion_matrix_make_miss.png', figsize=(5, 4.5))

## 10. Score counting

In [ ]:
expected_points = (probs * points_vec).sum(axis=1)

total_true_pts     = float(true_points.sum())
total_argmax_pts   = float(pred_points.sum())
total_expected_pts = float(expected_points.sum())
phantom_pts        = float(np.maximum(pred_points - true_points, 0).sum())
missed_pts         = float(np.maximum(true_points - pred_points, 0).sum())
signed_err_argmax  = total_argmax_pts   - total_true_pts
signed_err_soft    = total_expected_pts - total_true_pts
rel_err_argmax     = (signed_err_argmax / total_true_pts * 100) if total_true_pts > 0 else float('nan')
rel_err_soft       = (signed_err_soft   / total_true_pts * 100) if total_true_pts > 0 else float('nan')

score_summary = pd.DataFrame([
    ['Ground-truth points',                 f'{total_true_pts:8.1f}'],
    ['Argmax-predicted points',             f'{total_argmax_pts:8.1f}   (signed err {signed_err_argmax:+.1f}, rel err {rel_err_argmax:+6.2f}%)'],
    ['Expected-points (soft prediction)',   f'{total_expected_pts:8.1f}   (signed err {signed_err_soft:+.1f}, rel err {rel_err_soft:+6.2f}%)'],
    ['Phantom points (false positives)',    f'{phantom_pts:8.1f}'],
    ['Missed points (false negatives)',     f'{missed_pts:8.1f}'],
], columns=['', 'Value'])
try:
    display(score_summary)
except Exception:
    print(score_summary.to_string(index=False))

rows = []
for st_idx, st_name in enumerate(SHOT_TYPES):
    mask = true_shot_type == st_idx
    if mask.sum() == 0:
        continue
    rows.append({
        'shot_type':    st_name,
        'n_clips':      int(mask.sum()),
        'true_pts':     round(float(true_points[mask].sum()), 2),
        'argmax_pts':   round(float(pred_points[mask].sum()), 2),
        'expected_pts': round(float(expected_points[mask].sum()), 2),
    })
df_score = pd.DataFrame(rows)
df_score.loc['TOTAL'] = {
    'shot_type': 'TOTAL', 'n_clips': int(df_score['n_clips'].sum()),
    'true_pts':     round(float(df_score['true_pts'].sum()), 2),
    'argmax_pts':   round(float(df_score['argmax_pts'].sum()), 2),
    'expected_pts': round(float(df_score['expected_pts'].sum()), 2),
}
df_score.to_csv(REPORT_DIR / 'score_counter.csv', index=False)
print('\nPer shot-type subtotals:')
try:
    display(df_score)
except Exception:
    print(df_score.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4.5))
x = np.arange(len(df_score) - 1)
labels_plot = df_score['shot_type'].iloc[:-1].tolist()
width = 0.35
ax.bar(x - width/2, df_score['true_pts'].iloc[:-1].values,   width, label='ground truth', color='#1f77b4')
ax.bar(x + width/2, df_score['argmax_pts'].iloc[:-1].values, width, label='predicted',    color='#ff7f0e')
ax.set_xticks(x); ax.set_xticklabels(labels_plot)
ax.set_ylabel('Total points'); ax.set_title('OOD: total points by shot type')
ax.legend(); ax.grid(alpha=0.25, axis='y')
fig.tight_layout()
fig.savefig(REPORT_DIR / 'score_counter_bars.png', dpi=160, bbox_inches='tight')
plt.show(); plt.close(fig)